In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

#n_estimators_list = [100, 200, 300, 400, 500, 800]
#contamination_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, "auto"]
n_estimators_list = [100, 300, 500, 800]
contamination_list = [0.01, 0.05, 0.1, 0.5, "auto"]
max_train_samples = 5000


def contamination_to_folder_name(cont):
    if isinstance(cont, str):
        return f"contamination_{cont}"
    return f"contamination_{str(cont).replace('.', '_')}"


data_path = Path("../data_processed/NASA/nasa_battery_features_rolling.csv")
results_root = Path("../results/iForest_NASA_new")
results_root.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

df = df.dropna(subset=feature_cols + ["y_true_file"]).reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true_file"].astype(int)

X_normal = X[y_true == 0].copy()
X_anomaly = X[y_true == 1].copy()

summary_rows = []

for contamination in contamination_list:
    if isinstance(contamination, str):
        cont_str = contamination
    else:
        cont_str = f"{contamination:.4f}"

    scaler = StandardScaler()

    if len(X_normal) > max_train_samples:
        X_train_raw = X_normal.sample(n=max_train_samples, random_state=42)
    else:
        X_train_raw = X_normal.copy()

    X_train = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=feature_cols
    )

    X_test_normal = X_normal.copy()
    y_test_normal = pd.Series(np.zeros(len(X_test_normal), dtype=int))

    n_anomaly_test = min(2000, len(X_anomaly))
    X_test_anomaly = X_anomaly.sample(n=n_anomaly_test, random_state=42)
    y_test_anomaly = pd.Series(np.ones(len(X_test_anomaly), dtype=int))

    X_test = pd.concat([X_test_normal, X_test_anomaly], axis=0)
    y_test = pd.concat([y_test_normal, y_test_anomaly], axis=0)

    test_df = X_test.copy()
    test_df["y_true"] = y_test.values
    test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

    X_test = test_df.drop(columns=["y_true"])
    y_test = test_df["y_true"].astype(int)

    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=feature_cols
    )

    for n_estimators in n_estimators_list:
        exp_name = f"nasa_iforest_normaltrain_{n_estimators}_{cont_str}"
        results_dir = (
            results_root
            / f"n_estimators_{n_estimators}"
            / contamination_to_folder_name(contamination)
        )
        results_dir.mkdir(parents=True, exist_ok=True)

        model = IsolationForest(
            n_estimators=n_estimators,
            max_samples="auto",
            contamination=contamination,
            max_features=1.0,
            bootstrap=False,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train)

        #y_pred_raw = model.predict(X_test_scaled)
        #y_pred = (y_pred_raw == -1).astype(int)

        #scores = model.score_samples(X_test_scaled)
        scores = model.score_samples(X_test_scaled)
        
        threshold_percentile_list = [10, 30, 50, 70, 90]
        
        for threshold_percentile in threshold_percentile_list:
            threshold = np.percentile(scores, threshold_percentile)
            y_pred = (scores < threshold).astype(int)
        
            cm = confusion_matrix(y_test, y_pred)
        
            accuracy = accuracy_score(y_test, y_pred) * 100
            precision = precision_score(y_test, y_pred, zero_division=0) * 100
            recall = recall_score(y_test, y_pred, zero_division=0) * 100
            f1 = f1_score(y_test, y_pred, zero_division=0) * 100
        
            exp_name = (
                f"nasa_iforest_normaltrain_{n_estimators}"
                f"_{cont_str}"
                f"_p_{threshold_percentile}"
            )
        
            exp_dir = results_dir / f"percentile_{threshold_percentile}"
            exp_dir.mkdir(parents=True, exist_ok=True)
            
        #decisions = model.decision_function(X_test_scaled)

        df_results = X_test.copy()
        df_results["y_true"] = y_test.values
        #df_results["iforest_pred"] = y_pred_raw
        df_results["iforest_score"] = scores
        #df_results["iforest_decision"] = decisions
        df_results["threshold_percentile"] = threshold_percentile
        df_results["threshold"] = threshold
        df_results["is_anomaly"] = y_pred

        df_results.to_csv(results_dir / f"{exp_name}_results.csv", index=False)

        top_anomalies = df_results.sort_values("iforest_score").head(20)
        top_anomalies.to_csv(results_dir / f"{exp_name}_top20.csv", index=False)

        cm = confusion_matrix(y_test, y_pred)

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu Isolation Forest – NASA Battery", fontsize=12, pad=20)
        plt.savefig(results_dir / f"{exp_name}_metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(j, i, cm[i, j], ha="center", va="center", color=color)

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(results_dir / f"{exp_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        summary_rows.append({
            "n_estimators": n_estimators,
            "contamination": cont_str,
            "threshold_percentile": threshold_percentile,
            "threshold": threshold,
            "accuracy": round(accuracy, 2),
            "precision": round(precision, 2),
            "recall": round(recall, 2),
            "f1": round(f1, 2)
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
